# Experiment 10: End-to-End GAN Implementation (MNIST)

This notebook implements a **Generative Adversarial Network (GAN)** in detail with proper visualization, illustration, and demo.

## 1. Imports and Setup

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_style('whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

## 2. Load and Visualize Real Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)
print('x_test shape :', x_test.shape)
print('y_test shape :', y_test.shape)

In [ ]:
# Show sample real images
plt.figure(figsize=(10, 4))
for i in range(20):
    idx = np.random.randint(0, len(x_train))
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_train[idx], cmap='gray')
    plt.title(str(y_train[idx]), fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Real MNIST Images', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
plt.figure(figsize=(9, 4))
sns.countplot(x=y_train)
plt.title('Class Distribution in MNIST Training Set')
plt.xlabel('Digit class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Preprocessing for GAN

In [ ]:
# GAN generator uses tanh output, so we scale images to [-1, 1].
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

x_train = (x_train - 127.5) / 127.5
x_test = (x_test - 127.5) / 127.5

# Add channel dimension: (28, 28) -> (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print('Preprocessed x_train shape:', x_train.shape)
print('Pixel range:', x_train.min(), 'to', x_train.max())

In [ ]:
BUFFER_SIZE = x_train.shape[0]
BATCH_SIZE = 128

train_dataset = (
    tf.data.Dataset.from_tensor_slices(x_train)
    .shuffle(BUFFER_SIZE, seed=SEED)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print('Dataset pipeline created.')

## 4. Build Generator and Discriminator

In [ ]:
LATENT_DIM = 100

def build_generator(latent_dim=100):
    model = keras.Sequential(name='generator')
    model.add(layers.Input(shape=(latent_dim,)))
    model.add(layers.Dense(7 * 7 * 256, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((7, 7, 256)))

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Output in [-1, 1] because of tanh
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', activation='tanh'))
    return model

def build_discriminator(input_shape=(28, 28, 1)):
    model = keras.Sequential(name='discriminator')
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

generator = build_generator(LATENT_DIM)
discriminator = build_discriminator()

generator.summary()
discriminator.summary()

In [ ]:
# Quick generator sanity check
noise = tf.random.normal([16, LATENT_DIM])
generated_sample = generator(noise, training=False)

print('Generated sample shape:', generated_sample.shape)

plt.figure(figsize=(8, 4))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    img = (generated_sample[i, :, :, 0] + 1) / 2
    plt.imshow(img, cmap='gray')
    plt.axis('off')
plt.suptitle('Untrained Generator Output', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Define GAN Training Logic

In [ ]:
class GAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.d_loss_tracker = keras.metrics.Mean(name='d_loss')
        self.g_loss_tracker = keras.metrics.Mean(name='g_loss')

    @property
    def metrics(self):
        return [self.d_loss_tracker, self.g_loss_tracker]

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        if isinstance(real_images, tuple):
            real_images = real_images[0]

        batch_size = tf.shape(real_images)[0]

        # -----------------------------
        # Train Discriminator
        # -----------------------------
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        generated_images = self.generator(random_latent_vectors, training=True)

        combined_images = tf.concat([generated_images, real_images], axis=0)

        labels = tf.concat([
            tf.zeros((batch_size, 1)),
            tf.ones((batch_size, 1))
        ], axis=0)

        # Label smoothing / noise for stability
        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images, training=True)
            d_loss = self.loss_fn(labels, predictions)

        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        # -----------------------------
        # Train Generator
        # -----------------------------
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        misleading_labels = tf.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            fake_images = self.generator(random_latent_vectors, training=True)
            predictions = self.discriminator(fake_images, training=True)
            g_loss = self.loss_fn(misleading_labels, predictions)

        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        self.d_loss_tracker.update_state(d_loss)
        self.g_loss_tracker.update_state(g_loss)

        return {
            'd_loss': self.d_loss_tracker.result(),
            'g_loss': self.g_loss_tracker.result()
        }

In [ ]:
class GANMonitor(keras.callbacks.Callback):
    def __init__(self, num_img=16, latent_dim=100, out_dir='gan_epoch_images'):
        self.num_img = num_img
        self.latent_dim = latent_dim
        self.seed = tf.random.normal(shape=(num_img, latent_dim), seed=SEED)
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        generated = self.model.generator(self.seed, training=False)
        generated = (generated + 1) / 2.0

        fig = plt.figure(figsize=(6, 6))
        for i in range(self.num_img):
            plt.subplot(4, 4, i + 1)
            plt.imshow(generated[i, :, :, 0], cmap='gray')
            plt.axis('off')

        plt.suptitle(f'Generated Images at Epoch {epoch + 1}', fontsize=12)
        plt.tight_layout()
        filepath = os.path.join(self.out_dir, f'epoch_{epoch + 1:03d}.png')
        plt.savefig(filepath)
        plt.close(fig)

        if logs is not None:
            print(f"Epoch {epoch + 1:03d} | d_loss={logs.get('d_loss', 0):.4f} | g_loss={logs.get('g_loss', 0):.4f}")

## 6. Train GAN

In [ ]:
EPOCHS = 30

gan = GAN(discriminator=discriminator, generator=generator, latent_dim=LATENT_DIM)
gan.compile(
    d_optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    g_optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss_fn=keras.losses.BinaryCrossentropy()
)

monitor = GANMonitor(num_img=16, latent_dim=LATENT_DIM, out_dir='gan_epoch_images')
history = gan.fit(train_dataset, epochs=EPOCHS, callbacks=[monitor])

## 7. Visualize Training Dynamics

In [ ]:
hist = pd.DataFrame(history.history)

plt.figure(figsize=(10, 5))
plt.plot(hist['d_loss'], label='Discriminator Loss')
plt.plot(hist['g_loss'], label='Generator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Loss Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Show generated progression snapshots
saved_images = sorted(glob.glob('gan_epoch_images/epoch_*.png'))
print('Saved epoch snapshots:', len(saved_images))

if len(saved_images) > 0:
    show_ids = np.linspace(0, len(saved_images) - 1, num=min(6, len(saved_images)), dtype=int)
    plt.figure(figsize=(14, 8))
    for i, sid in enumerate(show_ids):
        img = plt.imread(saved_images[sid])
        plt.subplot(2, 3, i + 1)
        plt.imshow(img)
        epoch_num = sid + 1
        plt.title(f'Epoch {epoch_num}')
        plt.axis('off')
    plt.suptitle('Generator Progress Over Training', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No snapshot images found. Make sure training has run.')

## 8. Full Demo: Generate New Handwritten Digits

In [ ]:
def generate_images(model, n_images=25, latent_dim=100):
    z = tf.random.normal((n_images, latent_dim), seed=SEED)
    fake = model.generator(z, training=False).numpy()
    fake = (fake + 1) / 2.0
    return fake

generated_images = generate_images(gan, n_images=25, latent_dim=LATENT_DIM)

plt.figure(figsize=(8, 8))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(generated_images[i, :, :, 0], cmap='gray')
    plt.axis('off')
plt.suptitle('New Synthetic Digits Generated by GAN', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Save synthetic images for downstream usage/demo
N_SYNTHETIC = 2000
z = tf.random.normal((N_SYNTHETIC, LATENT_DIM), seed=SEED)
synthetic = gan.generator(z, training=False).numpy()
synthetic = ((synthetic + 1) / 2.0).astype('float32')

np.save('synthetic_mnist_gan.npy', synthetic)
print('Saved synthetic dataset:', synthetic.shape, '-> synthetic_mnist_gan.npy')

## 9. Utilization Example: Data Generation Pipeline

In [ ]:
# Utilization demo: compare real and generated sample statistics
real_sample = ((x_test[:2000] + 1) / 2.0).reshape(2000, -1)
fake_sample = synthetic[:2000].reshape(2000, -1)

real_mean = real_sample.mean(axis=1)
fake_mean = fake_sample.mean(axis=1)

plt.figure(figsize=(10, 5))
sns.kdeplot(real_mean, label='Real MNIST mean intensity', linewidth=2)
sns.kdeplot(fake_mean, label='Generated mean intensity', linewidth=2)
plt.title('Real vs Generated Pixel Intensity Profile')
plt.xlabel('Mean pixel intensity per image')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

print('Real sample mean intensity :', round(float(real_mean.mean()), 4))
print('Generated mean intensity   :', round(float(fake_mean.mean()), 4))

In [ ]:
# Discriminator confidence check on real vs fake
real_scores = gan.discriminator(x_test[:1024], training=False).numpy().ravel()
fake_scores = gan.discriminator((synthetic[:1024] * 2.0 - 1.0), training=False).numpy().ravel()

plt.figure(figsize=(10, 4))
sns.kdeplot(real_scores, label='D(real)', linewidth=2)
sns.kdeplot(fake_scores, label='D(fake)', linewidth=2)
plt.title('Discriminator Output Distribution')
plt.xlabel('Discriminator score')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

print('Average D(real):', round(float(real_scores.mean()), 4))
print('Average D(fake):', round(float(fake_scores.mean()), 4))

## 10. Latent Space Interpolation Demo

In [ ]:
# Interpolate between two random latent vectors to visualize smooth transitions
z1 = tf.random.normal((1, LATENT_DIM), seed=SEED)
z2 = tf.random.normal((1, LATENT_DIM), seed=SEED + 1)

alphas = np.linspace(0, 1, 12)
interpolated = []
for a in alphas:
    z = (1 - a) * z1 + a * z2
    img = gan.generator(z, training=False).numpy()[0, :, :, 0]
    interpolated.append((img + 1) / 2.0)

plt.figure(figsize=(14, 2.8))
for i, img in enumerate(interpolated):
    plt.subplot(1, len(interpolated), i + 1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title(f'{alphas[i]:.1f}', fontsize=8)

plt.suptitle('Latent Interpolation: Smooth Morphing Between Generated Digits', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Optional: Create GIF from Epoch Snapshots

In [ ]:
# Optional convenience block to create a training GIF.
try:
    import imageio.v2 as imageio

    files = sorted(glob.glob('gan_epoch_images/epoch_*.png'))
    if len(files) > 0:
        frames = [imageio.imread(f) for f in files]
        imageio.mimsave('gan_training_progress.gif', frames, duration=0.45)
        print('GIF created: gan_training_progress.gif')
    else:
        print('No epoch images found for GIF creation.')
except Exception as e:
    print('GIF creation skipped. Install imageio if needed.')
    print('Reason:', e)

## 12. Conclusion

You completed an end-to-end GAN demo with:
- Real dataset loading and preprocessing
- Generator and discriminator design
- Custom adversarial training loop
- Epoch-wise visualization of generated results
- Loss curves and quality diagnostics
- Utilization example by creating and saving synthetic images
- Latent space interpolation for interpretability

This is a full practical GAN workflow you can adapt to Fashion-MNIST, CIFAR-10, or your own custom image datasets.